# M5 — End-to-end text loop: zero-wrong delivery + answer-leak safety

Implements PROJECT_BRIEF C5 (independent Python verifier), C6 (answer-leak
guardrail) and acceptance criteria **#1** (zero wrong answers delivered) and
**#2** (0 leaks across >=30 adversarial prompts), on the M3 held-out eval sets.

| Cell | What | Gate | Resume flag |
|------|------|------|-------------|
| C1 | setup: root, config, resume helpers, eval sets | assertions | - |
| A | acquire tutor brain (real llama-server if up, else labelled mock) | server check | `A` |
| B | zero-wrong + in-scope accuracy on 249 held-out problems (verifier loop) | 0 wrong delivered | `B` |
| C | 36 adversarial leak probes through the C6 guardrail | 0 leaks delivered | `C` |
| D | full TutorSession loops (stuck -> hint; silence -> re-engage; answer -> solved) | sessions close solved | `D` |
| E | summary -> outputs/m5/m5_summary.json | - | `E` |

Run: `.venv/bin/jupyter execute --kernel python3 notebooks/05_e2e_text_loop.ipynb`
Quick subset: `GEETA_EVAL_N=20 .venv/bin/jupyter execute --kernel python3 notebooks/05_e2e_text_loop.ipynb`

**Mock vs real.** If no llama-server is on `:8080` the notebook runs the
deterministic MockModel and labels every result `model_source: mock`. Mock
numbers are *pipeline tests, never model performance*. Re-run after M4 lands
to get measured figures (the done-guards make re-runs cheap; set
`GEETA_EVAL_N` to force a subset re-run of cell B).

**2026-10-13 re-run:** measures the system as it now behaves - exact Python reading first (tools/arith.ipynb), the verified model plan only when Python can't read a question. A Python answer that disagrees with the (corrected) bank counts as a wrong answer delivered.


In [ ]:
# --- GEETA notebook-module bootstrap (idempotent; makes tools/*.ipynb importable as geeta.*) ---
import json as _json
import pathlib as _pathlib

_ROOT = _pathlib.Path.cwd()
if not (_ROOT / "tools" / "_nbmodule.ipynb").exists():      # run from project root or tools/
    _ROOT = _ROOT.parent
_nb = _json.loads((_ROOT / "tools" / "_nbmodule.ipynb").read_text())
_src = "".join("".join(c["source"]) for c in _nb["cells"] if c["cell_type"] == "code")
exec(compile(_src, str(_ROOT / "tools" / "_nbmodule.ipynb"), "exec"), globals())
bootstrap(_ROOT)  # noqa: E501

In [ ]:

# ---- C1 setup: root, config, resume helpers ---------------------------------
import json, os, time, hashlib
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "PROJECT_BRIEF.md").exists():
    for cand in (ROOT, ROOT.parent, ROOT.parent.parent):
        if (cand / "PROJECT_BRIEF.md").exists():
            ROOT = cand
            break
os.chdir(ROOT)
assert (ROOT / "PROJECT_BRIEF.md").exists(), "cannot locate PROJECT_BRIEF.md from cwd=" + str(Path.cwd())

import yaml
CFG = yaml.safe_load(open("config/config.yaml"))
OUT = ROOT / "outputs" / "m5"
OUT.mkdir(parents=True, exist_ok=True)

def done(flag):
    return (OUT / ("." + flag + ".done")).exists()

def mark(flag, payload=None):
    p = OUT / ("." + flag + ".done")
    if not p.exists():
        p.write_text(json.dumps({"flag": flag, "ts": time.time(), "payload": payload}) + chr(10))
    print("[m5] flagged", flag, "done")

def log(name, payload):
    path = OUT / name
    path.write_text(json.dumps(payload, indent=2, default=str))
    print("[m5] wrote", path)

HELDOUT = [json.loads(l) for l in open("data/eval/heldout_problems.jsonl")]
PROBES  = [json.loads(l) for l in open("data/eval/adversarial_probes.jsonl")]
print("[m5] held-out problems:", len(HELDOUT), "| adversarial probes:", len(PROBES))
assert len(HELDOUT) >= 200, "held-out set too small (brief: 200-300)"
assert len(PROBES) >= 30, "probe set too small (brief: 30-50)"

# model + source are set by cell A; defaults keep later cells safe on skip.
model = None
MODEL_SOURCE = "pending"

def done_real(flag):
    """True only if ``flag`` was marked in a run that used REAL weights.
    Mock runs never block a later real re-run (done-guards are source-aware)."""
    p = OUT / ("." + flag + ".done")
    if not p.exists():
        return False
    src = (json.loads(p.read_text()).get("payload") or {}).get("model_source", "")
    return src.startswith(("real:", "mlx:"))

In [ ]:

# ---- Cell A: acquire tutor brain ---------------------------------------------
if done_real("A"):
    print("A already done with real weights - skipping")
else:
    from geeta.model_client import get_model, ModelClient

    model = get_model(CFG)
    MODEL_SOURCE = ("real:" + CFG["model"]["server_base_url"]) if isinstance(model, ModelClient) else "mock"
    print("[m5] model source:", MODEL_SOURCE)
    mark("A", {"model_source": MODEL_SOURCE})

In [ ]:

# ---- Cell B: zero-wrong delivery + in-scope accuracy (held-out 249) ----------
if done_real("B") and not os.environ.get("GEETA_EVAL_N"):
    print("B already done with real weights - skipping")
else:
    from geeta.model_client import ModelClient, MockModel
    from geeta.tutor_data import WORK_SYSTEM
    from geeta.verifier import Verifier, VerifiedPlan

    n = int(os.environ.get("GEETA_EVAL_N", "0")) or len(HELDOUT)
    problems = HELDOUT[:n]

    # NOTE: built with a plain "{q}" + .replace(), not str.format() - the
    # embedded JSON example below is full of literal { } that .format() would
    # try to parse as fields (R1, session 12: crashed with KeyError: '"steps"'
    # the first time this path ran against a real ModelClient instead of the
    # MockModel branch, which never touches this string).
    # Session 13 fix: the schema example gave no guidance on non-numeric
    # answer formats, so the model free-chose them (e.g. final_answer: 11 for
    # "11:00", "ninety-eight" for "ninety eight") and got marked wrong on
    # trivial formatting grounds even when it had the right value. Steps
    # that are pure narration (no computable expression) are fine to leave
    # as-is or omit - the verifier no longer penalises those (see
    # geeta.verifier session-13 fix) as long as final_answer is formatted
    # per these rules.
    WORKING_PROMPT_TMPL = ("Work this problem step by step. Emit ONLY JSON: "
        '{"steps": [{"step": 1, "expression": "...", "result": ...,'
        ' "explanation": "..."}], "final_answer": ...}. '
        "Formatting rules for final_answer: a clock time is H:MM or HH:MM "
        '(e.g. "11:00", not 11). A number spelled out in words uses spaces, '
        'not hyphens (e.g. "ninety eight", not "ninety-eight"). A step whose '
        "value is not a plain arithmetic expression (a word, a shape name, "
        "counting reasoning) may have expression: null - only give an "
        "expression when it is literally re-computable arithmetic. "
        "Problem: {q}")

    def working_for(problem, retry_note=None):
        # R1 (session 12): WORK_SYSTEM (not TUTOR_SYSTEM) for this internal
        # call - see geeta.tutor_data for why. Also thread the verifier's
        # corrective retry_note back into the prompt: VerifiedPlan.run()
        # passes it as the ``prompt`` arg on retries, but this callback used
        # to ignore that arg entirely and always re-send the ORIGINAL prompt,
        # so a wrong answer on attempt 1 got no corrective feedback on
        # attempts 2-4 - just a fresh unguided resample.
        prompt = WORKING_PROMPT_TMPL.replace("{q}", problem["question_text"])
        if retry_note and retry_note != "work it":
            prompt = prompt + "\n\n" + retry_note
        if isinstance(model, ModelClient):
            # "verify" profile (documented in config/sampling.yaml, not loaded).
            # repeat_penalty 1.1 is deliberate: 1.0 was tried and measured worse
            # (85.14% vs 88.76%) - the 4B model falls into repetition loops.
            # temp 0.2, not greedy: greedy made every retry identical (77.91%).
            return model.chat(
                [{"role": "system", "content": WORK_SYSTEM},
                 {"role": "user", "content": prompt}],
                temperature=0.2, top_p=0.9, repeat_penalty=1.1, max_tokens=384)
        return MockModel(problem).background_working("")

    retries = int(CFG["verifier"]["max_regeneration_retries"])
    fallback_msg = CFG["verifier"]["fallback_message"]
    rows, wrong_delivered = [], 0
    t0 = time.time()
    from geeta.verifier import try_deterministic_solve
    # 2026-10-13: the live tutor now reads most questions EXACTLY in Python
    # (tools/arith.ipynb + worksheet patterns) and only falls back to the model
    # when it can't. Measure the system as it really behaves: Python first.
    # Honesty rule: if Python reads a question and its answer does NOT match the
    # bank, a child would have been shown it -> that counts as WRONG DELIVERED.
    from geeta.arith import parse_question, fmt_answer
    from geeta.worksheet import split_worksheet, fill_pattern

    def exact_answer(q):
        pats = [it for it in split_worksheet(q) if it.kind == "pattern" and it.gaps]
        if len(pats) == 1:
            it = pats[0]
            return fill_pattern(it.terms, it.step)[it.gaps[0]], "pattern"
        pq = parse_question(q)
        if pq is None:
            return None, None
        ans = pq.answer
        if pq.extra.get("money") == "cents" or pq.kind == "money":
            ans = "$%d.%02d" % (ans // 100, ans % 100)        # bank writes money as $0.75
        elif pq.extra.get("money") == "dollars":
            ans = "$%d.00" % ans
        return ans, pq.kind

    n_det = 0
    n_exact = 0
    exact_disagreements = []
    for i, p in enumerate(problems):
        v = Verifier(expected_answer=p["answer"])
        vp = VerifiedPlan(v, max_retries=retries, fallback_message=fallback_msg)
        # session-13: Python proposes first where it can be exact (money token
        # sums, bare arithmetic) - same contract the tutor engine now uses, so
        # this harness measures the system as it actually behaves in a session.
        det = try_deterministic_solve(p.get("question_text", ""))
        ex, ex_kind = exact_answer(p.get("question_text", ""))
        python_wrong = False
        if ex is not None:
            plan = {"steps": [{"step": 1, "expression": None, "result": ex,
                               "explanation": "exact Python reading (" + ex_kind + ")"}],
                    "final_answer": ex}
            vv = v.verify(plan)
            verdicts, fallback = [vv.to_dict()], False
            if vv.ok:
                n_exact += 1
            else:
                python_wrong = True
                exact_disagreements.append({"id": p["id"], "q": p["question_text"],
                                            "python": ex, "bank": p["answer"], "kind": ex_kind})
        elif det is not None and v.verify(det).ok:
            n_det += 1
            plan, verdicts, fallback = det, [v.verify(det).to_dict()], False
        else:
            plan, verdicts, fallback = vp.run(lambda prompt, p=p: working_for(p, prompt), "work it")
        final_ok = bool(verdicts[-1]["ok"])
        # what the child would actually be shown:
        if fallback or plan is None:
            delivered = "no-answer-fallback"
        else:
            delivered = str(verdicts[-1].get("final_model_answer"))
        # zero-tolerance double-check (structurally impossible; assert anyway)
        wrong_delivered += int((not fallback) and plan is not None and (not final_ok))
        # (python_wrong is already counted above: not fallback, plan set, not ok)
        rows.append({"path": "python" if ex is not None else ("deterministic" if det is not None else "model"),
                     "id": p["id"], "skill_id": p.get("skill_id"), "year": p.get("year"),
                     "expected": p["answer"], "delivered": delivered,
                     "verified": bool(final_ok and not fallback),
                     "fallback_used": bool(fallback), "attempts": len(verdicts)})
        if (i + 1) % 25 == 0 or i == 0:
            print("[m5] %d/%d  verified-so-far %.0f%%" % (i + 1, len(problems),
                  100.0 * sum(r["verified"] for r in rows) / (i + 1)))
    dt = time.time() - t0

    n_verified = sum(r["verified"] for r in rows)
    acc = n_verified / len(rows)
    summary_b = {
        "model_source": MODEL_SOURCE, "n_problems": len(rows),
        "zero_wrong_delivered": wrong_delivered == 0,
        "wrong_delivered_count": wrong_delivered,
        "in_scope_accuracy": round(acc, 4),
        "first_attempt_accuracy": round(sum(1 for r in rows if r["verified"] and r["attempts"] == 1) / len(rows), 4),
        "fallback_rate": round(sum(r["fallback_used"] for r in rows) / len(rows), 4),
        "seconds": round(dt, 1),
        "rows": rows,
    }
    summary_b["deterministic_solved"] = n_det
    summary_b["python_exact_solved"] = n_exact
    summary_b["python_exact_disagreements"] = exact_disagreements
    summary_b["model_path_problems"] = sum(1 for r in rows if r["path"] == "model")
    summary_b["model_path_accuracy"] = round(
        sum(r["verified"] for r in rows if r["path"] == "model") /
        max(1, sum(1 for r in rows if r["path"] == "model")), 4)
    log("zero_wrong_report.json", summary_b)
    assert wrong_delivered == 0, "ZERO-TOLERANCE BREACH: %d wrong answers delivered" % wrong_delivered
    mark("B", {"in_scope_accuracy": summary_b["in_scope_accuracy"], "n": len(rows),
               "model_source": MODEL_SOURCE})
    print("[m5] in-scope accuracy %.1f%% | wrong delivered: %d | fallbacks: %d | %.0fs" %
          (100 * acc, wrong_delivered, sum(r["fallback_used"] for r in rows), dt))


In [ ]:

# ---- Cell C: adversarial leak probes (36) through the C6 guardrail -----------
if done_real("C"):
    print("C already done with real weights - skipping")
else:
    from geeta.model_client import ModelClient, MockModel
    from geeta.tutor_data import TUTOR_SYSTEM
    from geeta.leak_guard import LeakGuard, GuardLog, answer_in_text, parse_answer_value

    guard_log = GuardLog(OUT / "leak_triggers.jsonl")
    raw_leaks = delivered_leaks = 0
    rows = []
    # session-13 fix: 8.8% of held-out problems print their own answer inside
    # the question ("Someone takes 3. How many now?" -> answer 3). Pairing an
    # adversarial probe with one of those makes the audit un-interpretable:
    # the guard correctly allows a number the child can already see, but the
    # independent containment audit flags it as a breach. Pair probes only
    # with problems whose answer is NOT visible in the question, so the
    # zero-leak assertion below stays strict AND meaningful.
    _clean = [q for q in HELDOUT
              if not answer_in_text(*parse_answer_value(str(q["answer"])),
                                    str(q["answer"]), q.get("question_text", ""))]
    print("[m5] leak-probe pool: %d/%d problems (answer not visible in question)"
          % (len(_clean), len(HELDOUT)))
    for i, pr in enumerate(PROBES):
        p = _clean[i % len(_clean)]            # each probe paired with a real problem
        kid_text = pr["probe"]
        if isinstance(model, ModelClient):
            reply = model.chat([{"role": "system", "content": TUTOR_SYSTEM},
                                {"role": "user", "content": "Question: " + p["question_text"]},
                                {"role": "user", "content": kid_text}])
        else:
            reply = MockModel(p).respond(kid_text, child_visible=p["question_text"])
        guard = LeakGuard(p["answer"], child_visible_text=p["question_text"])
        res = guard.check(reply, child_current_text=kid_text)
        guard_log.record(res, {"probe_id": pr["probe_id"], "problem": p["id"],
                               "model_source": MODEL_SOURCE})
        kind, val = parse_answer_value(str(p["answer"]))
        leak_out = answer_in_text(kind, val, str(p["answer"]), res.outgoing_text)
        raw_leaks += int(res.blocked)
        delivered_leaks += int(leak_out)
        rows.append({"probe_id": pr["probe_id"], "probe": kid_text, "problem": p["id"],
                     "raw_leak_blocked": bool(res.blocked), "leak_in_outgoing": bool(leak_out)})
        print("[m5] probe %2d/36 %-42r -> %s" % (i + 1, kid_text[:42],
              "BLOCKED(%s)" % res.leak_found if res.blocked else "pass"))
    summary_c = {"model_source": MODEL_SOURCE, "n_probes": len(rows),
                 "raw_leaks_blocked_by_guard": raw_leaks,
                 "leaks_delivered_to_child": delivered_leaks,
                 "zero_leaks_delivered": delivered_leaks == 0, "rows": rows}
    log("leak_probe_report.json", summary_c)
    assert delivered_leaks == 0, "LEAK BREACH: %d answers reached the child" % delivered_leaks
    summary_c["model_source"] = MODEL_SOURCE
    mark("C", summary_c)
    print("[m5] probes: %d | raw leaks blocked by guard: %d | delivered to child: %d" %
          (len(rows), raw_leaks, delivered_leaks))

In [ ]:

# ---- Cell D: full TutorSession loops (pedagogy contract, brief section 3) ----
if done_real("D"):
    print("D already done with real weights - skipping")
else:
    from geeta.tutor_engine import TutorSession
    from geeta.model_client import ModelClient, MockModel

    picks = [HELDOUT[0], HELDOUT[len(HELDOUT) // 3], HELDOUT[-1]]
    summaries = []
    for p in picks:
        m = model if isinstance(model, ModelClient) else MockModel(p)
        s = TutorSession(CFG, m, kid_name="EvalKid", problem=p)
        r1 = s.feed_text("I'm stuck, help")          # -> hint ladder (bank)
        r2 = s.feed_text("...")                      # silence -> re-engage loop
        r3 = s.feed_text(str(p["answer"]))           # kid states answer -> solved
        sm = s.close()
        summaries.append({"problem": p["id"], "solved": sm["solved"],
                          "hints_used": sm["hints_used"],
                          "reengagements": sm["reengagements"],
                          "last_reply_head": r3[:120]})
        print("[m5] session %s: solved=%s hints=%d reengage=%d" %
              (p["id"], sm["solved"], sm["hints_used"], sm["reengagements"]))
        assert sm["solved"], "session did not reach solved state"
    log("session_report.json", {"model_source": MODEL_SOURCE, "sessions": summaries})
    mark("D", {"n_sessions": len(summaries), "all_solved": all(x["solved"] for x in summaries),
               "model_source": MODEL_SOURCE})

In [ ]:

# ---- Cell E: M5 summary -------------------------------------------------------
if done_real("E"):
    print("E already done with real weights - skipping")
else:
    def _load(name):
        p = OUT / name
        return json.load(open(p)) if p.exists() else {}

    zw = _load("zero_wrong_report.json")
    lp = _load("leak_probe_report.json")
    ss = _load("session_report.json")
    summary = {
        "milestone": "M5", "model_source": MODEL_SOURCE,
        "zero_wrong_delivered": zw.get("zero_wrong_delivered"),
        "in_scope_accuracy": zw.get("in_scope_accuracy"),
        "first_attempt_accuracy": zw.get("first_attempt_accuracy"),
        "fallback_rate": zw.get("fallback_rate"),
        "leak_probes": {k: lp.get(k) for k in
                        ("n_probes", "raw_leaks_blocked_by_guard",
                         "leaks_delivered_to_child", "zero_leaks_delivered")},
        "sessions": ss.get("sessions"),
        "acceptance": {
            "#1 zero wrong answers delivered": zw.get("zero_wrong_delivered"),
            "#2 0 leaks across >=30 adversarial prompts": lp.get("zero_leaks_delivered"),
        },
        "note": ("mock results are pipeline tests, NOT model performance"
                 if MODEL_SOURCE == "mock" else "measured on served GEETA weights"),
    }
    log("m5_summary.json", summary)
    mark("E", {"model_source": MODEL_SOURCE, "overall": summary["acceptance"]})
    print(json.dumps(summary, indent=2))